In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [3]:
clean_df = pd.read_csv("../data/processed/cleaned_online_retail_with_skew_features.csv")
customer_df = pd.read_csv("../data/processed/customer_sales_for_feature_engineering.csv")

clean_df["invoice_date"] = pd.to_datetime(clean_df["invoice_date"])
clean_df["date"] = pd.to_datetime(clean_df["date"])

customer_df["invoice_date"] = pd.to_datetime(customer_df["invoice_date"])
customer_df["date"] = pd.to_datetime(customer_df["date"])

print("Clean sales data:", clean_df.shape)
print("Customer sales data:", customer_df.shape)

clean_df.head()

Clean sales data: (524878, 24)
Customer sales data: (392692, 18)


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,customer_id_missing,is_cancelled,...,day,day_of_week,hour,year_month,log_quantity,log_unit_price,log_total_amount,quantity_capped,unit_price_capped,total_amount_capped
0,536365,85123A,white hanging heart t-light holder,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,0,...,1,Wednesday,8,2010-12,1.945910,1.266948,2.791165,6.0,2.55,15.30
1,536365,71053,white metal lantern,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,0,...,1,Wednesday,8,2010-12,1.945910,1.479329,3.060583,6.0,3.39,20.34
2,536365,84406B,cream cupid hearts coat hanger,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,0,...,1,Wednesday,8,2010-12,2.197225,1.321756,3.135494,8.0,2.75,22.00
3,536365,84029G,knitted union flag hot water bottle,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,0,...,1,Wednesday,8,2010-12,1.945910,1.479329,3.060583,6.0,3.39,20.34
4,536365,84029E,red woolly hottie white heart.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,0,...,1,Wednesday,8,2010-12,1.945910,1.479329,3.060583,6.0,3.39,20.34


In [4]:
required_clean_cols = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
    "total_amount",
    "date",
    "year",
    "month",
    "day",
    "day_of_week",
    "hour",
    "year_month",
    "log_total_amount",
    "total_amount_capped"
]

missing_cols = [
    col for col in required_clean_cols
    if col not in clean_df.columns
]

if len(missing_cols) == 0:
    print("All required columns are available.")
else:
    print("Missing columns:", missing_cols)

All required columns are available.


In [5]:
print("Missing values in clean_df:")
print(clean_df.isnull().sum())

print("\nMissing values in customer_df:")
print(customer_df.isnull().sum())

print("\nDuplicate rows in clean_df:", clean_df.duplicated().sum())
print("Duplicate rows in customer_df:", customer_df.duplicated().sum())

print("\nDate range:")
print(clean_df["invoice_date"].min(), "to", clean_df["invoice_date"].max())

Missing values in clean_df:
invoice_no                  0
stock_code                  0
description                 0
quantity                    0
invoice_date                0
unit_price                  0
customer_id            132186
country                     0
customer_id_missing         0
is_cancelled                0
total_amount                0
date                        0
year                        0
month                       0
day                         0
day_of_week                 0
hour                        0
year_month                  0
log_quantity                0
log_unit_price              0
log_total_amount            0
quantity_capped             0
unit_price_capped           0
total_amount_capped         0
dtype: int64

Missing values in customer_df:
invoice_no             0
stock_code             0
description            0
quantity               0
invoice_date           0
unit_price             0
customer_id            0
country                0
custome

Engineering Features....

In [7]:
sales_features = clean_df.copy()

sales_features["invoice_week"] = sales_features["invoice_date"].dt.isocalendar().week.astype(int)
sales_features["invoice_quarter"] = sales_features["invoice_date"].dt.quarter
sales_features["invoice_dayofweek_num"] = sales_features["invoice_date"].dt.dayofweek
sales_features["is_weekend"] = sales_features["invoice_dayofweek_num"].isin([5, 6]).astype(int)

sales_features["revenue_per_unit"] = sales_features["total_amount"] / sales_features["quantity"]

sales_features.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,customer_id_missing,is_cancelled,...,log_unit_price,log_total_amount,quantity_capped,unit_price_capped,total_amount_capped,invoice_week,invoice_quarter,invoice_dayofweek_num,is_weekend,revenue_per_unit
0,536365,85123A,white hanging heart t-light holder,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,0,...,1.266948,2.791165,6.0,2.55,15.30,48,4,2,0,2.55
1,536365,71053,white metal lantern,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,0,...,1.479329,3.060583,6.0,3.39,20.34,48,4,2,0,3.39
2,536365,84406B,cream cupid hearts coat hanger,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,0,...,1.321756,3.135494,8.0,2.75,22.00,48,4,2,0,2.75
3,536365,84029G,knitted union flag hot water bottle,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,0,...,1.479329,3.060583,6.0,3.39,20.34,48,4,2,0,3.39
4,536365,84029E,red woolly hottie white heart.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,0,...,1.479329,3.060583,6.0,3.39,20.34,48,4,2,0,3.39


In [8]:
product_features = (
    clean_df.groupby(["stock_code", "description"])
    .agg(
        total_quantity_sold=("quantity", "sum"),
        total_revenue=("total_amount", "sum"),
        total_revenue_capped=("total_amount_capped", "sum"),
        total_orders=("invoice_no", "nunique"),
        unique_customers=("customer_id", "nunique"),
        avg_unit_price=("unit_price", "mean"),
        avg_quantity_per_order=("quantity", "mean"),
        first_sale_date=("invoice_date", "min"),
        last_sale_date=("invoice_date", "max")
    )
    .reset_index()
)

product_features["product_active_days"] = (
    product_features["last_sale_date"] - product_features["first_sale_date"]
).dt.days

product_features["avg_daily_quantity_sold"] = (
    product_features["total_quantity_sold"] / (product_features["product_active_days"] + 1)
)

product_features["revenue_per_order"] = (
    product_features["total_revenue"] / product_features["total_orders"]
)

product_features = product_features.sort_values(
    by="total_revenue",
    ascending=False
)

product_features.head()

,stock_code,description,total_quantity_sold,total_revenue,total_revenue_capped,total_orders,unique_customers,avg_unit_price,avg_quantity_per_order,first_sale_date,last_sale_date,product_active_days,avg_daily_quantity_sold,revenue_per_order
4147,DOT,dotcom postage,706,206248.77,107460.99,706,1,292.137068,1.000000,2010-12-01 14:32:00,2011-12-09 10:26:00,372,1.892761,292.137068
1340,22423,regency cakestand 3 tier,13851,174156.54,124719.01,1988,881,13.983936,6.901345,2010-12-01 12:27:00,2011-12-09 10:23:00,372,37.134048,87.603893
2665,23843,"paper craft , little birdie",80995,168469.60,183.60,1,1,2.080000,80995.000000,2011-12-09 09:15:00,2011-12-09 09:15:00,0,80995.000000,168469.600000
3637,85123A,white hanging heart t-light holder,37580,104284.24,73810.85,2189,856,3.116159,16.746881,2010-12-01 08:26:00,2011-12-08 19:55:00,372,100.750670,47.640128
2874,47566,party bunting,18283,99445.23,73923.43,1685,708,5.797928,10.761036,2010-12-03 11:27:00,2011-12-09 10:23:00,370,49.280323,59.017941


In [10]:
sales_features.to_csv("../data/processed/sales_features.csv", index=False)
product_features.to_csv("../data/processed/product_features.csv", index=False)

print("sales_features:", sales_features.shape)
print("product_features:", product_features.shape)

print("Sales and product features saved successfully.")

sales_features: (524878, 29)
product_features: (4158, 14)
Sales and product features saved successfully.


23843 - paper craft, little birdie

is a one-time bulk purchase:

total_orders = 1
quantity = 80995
active_days = 0

So later we will not use this product for product-level demand forecasting. But it can stay in product features.


In [11]:
snapshot_date = customer_df["invoice_date"].max() + pd.Timedelta(days=1)

rfm_features = (
    customer_df.groupby("customer_id")
    .agg(
        recency=("invoice_date", lambda x: (snapshot_date - x.max()).days),
        frequency=("invoice_no", "nunique"),
        monetary=("total_amount", "sum"),
        monetary_capped=("total_amount", lambda x: np.minimum(x, x.quantile(0.99)).sum()),
        total_quantity=("quantity", "sum"),
        unique_products=("stock_code", "nunique"),
        avg_order_value=("total_amount", "mean"),
        first_purchase=("invoice_date", "min"),
        last_purchase=("invoice_date", "max")
    )
    .reset_index()
)

rfm_features.head()

,customer_id,recency,frequency,monetary,monetary_capped,total_quantity,unique_products,avg_order_value,first_purchase,last_purchase
0,12346,326,1,77183.60,77183.600,74215,1,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00
1,12347,2,7,4310.00,4166.600,2458,103,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00
2,12348,75,4,1797.24,1770.240,2341,22,57.975484,2010-12-16 19:09:00,2011-09-25 13:13:00
3,12349,19,1,1757.55,1587.486,631,73,24.076027,2011-11-21 09:51:00,2011-11-21 09:51:00
4,12350,310,1,334.40,332.032,197,17,19.670588,2011-02-02 16:01:00,2011-02-02 16:01:00


In [12]:
rfm_features["customer_lifetime_days"] = (
    rfm_features["last_purchase"] - rfm_features["first_purchase"]
).dt.days

rfm_features["purchase_frequency_per_month"] = (
    rfm_features["frequency"] / ((rfm_features["customer_lifetime_days"] + 1) / 30)
)

rfm_features["avg_quantity_per_order"] = (
    rfm_features["total_quantity"] / rfm_features["frequency"]
)

rfm_features["monetary_per_order"] = (
    rfm_features["monetary"] / rfm_features["frequency"]
)

rfm_features["product_diversity_ratio"] = (
    rfm_features["unique_products"] / rfm_features["unique_products"].max()
)

rfm_features.head()

,customer_id,recency,frequency,monetary,monetary_capped,total_quantity,unique_products,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency_per_month,avg_quantity_per_order,monetary_per_order,product_diversity_ratio
0,12346,326,1,77183.60,77183.600,74215,1,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00,0,30.000000,74215.000000,77183.600000,0.000560
1,12347,2,7,4310.00,4166.600,2458,103,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00,365,0.573770,351.142857,615.714286,0.057639
2,12348,75,4,1797.24,1770.240,2341,22,57.975484,2010-12-16 19:09:00,2011-09-25 13:13:00,282,0.424028,585.250000,449.310000,0.012311
3,12349,19,1,1757.55,1587.486,631,73,24.076027,2011-11-21 09:51:00,2011-11-21 09:51:00,0,30.000000,631.000000,1757.550000,0.040851
4,12350,310,1,334.40,332.032,197,17,19.670588,2011-02-02 16:01:00,2011-02-02 16:01:00,0,30.000000,197.000000,334.400000,0.009513


In [13]:
rfm_features["r_score"] = pd.qcut(
    rfm_features["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1],
    duplicates="drop"
).astype(int)

rfm_features["f_score"] = pd.qcut(
    rfm_features["frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5],
    duplicates="drop"
).astype(int)

rfm_features["m_score"] = pd.qcut(
    rfm_features["monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5],
    duplicates="drop"
).astype(int)

rfm_features["rfm_score"] = (
    rfm_features["r_score"] +
    rfm_features["f_score"] +
    rfm_features["m_score"]
)

rfm_features.head()

,customer_id,recency,frequency,monetary,monetary_capped,total_quantity,unique_products,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency_per_month,avg_quantity_per_order,monetary_per_order,product_diversity_ratio,r_score,f_score,m_score,rfm_score
0,12346,326,1,77183.60,77183.600,74215,1,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00,0,30.000000,74215.000000,77183.600000,0.000560,1,1,5,7
1,12347,2,7,4310.00,4166.600,2458,103,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00,365,0.573770,351.142857,615.714286,0.057639,5,5,5,15
2,12348,75,4,1797.24,1770.240,2341,22,57.975484,2010-12-16 19:09:00,2011-09-25 13:13:00,282,0.424028,585.250000,449.310000,0.012311,2,4,4,10
3,12349,19,1,1757.55,1587.486,631,73,24.076027,2011-11-21 09:51:00,2011-11-21 09:51:00,0,30.000000,631.000000,1757.550000,0.040851,4,1,4,9
4,12350,310,1,334.40,332.032,197,17,19.670588,2011-02-02 16:01:00,2011-02-02 16:01:00,0,30.000000,197.000000,334.400000,0.009513,1,1,2,4


In [14]:
def assign_rfm_segment(score):
    if score >= 13:
        return "Top Value"
    elif score >= 10:
        return "High Value"
    elif score >= 7:
        return "Medium Value"
    else:
        return "Low Value"

rfm_features["rfm_segment"] = rfm_features["rfm_score"].apply(assign_rfm_segment)

rfm_features["rfm_segment"].value_counts()

rfm_segment
Low Value       1307
Medium Value    1088
High Value      1010
Top Value        933
Name: count, dtype: int64

In [15]:
rfm_features.to_csv("../data/processed/customer_rfm_features.csv", index=False)

print("customer_rfm_features:", rfm_features.shape)
print("RFM features saved successfully.")

rfm_features.head()

customer_rfm_features: (4338, 20)
RFM features saved successfully.


,customer_id,recency,frequency,monetary,monetary_capped,total_quantity,unique_products,avg_order_value,first_purchase,last_purchase,customer_lifetime_days,purchase_frequency_per_month,avg_quantity_per_order,monetary_per_order,product_diversity_ratio,r_score,f_score,m_score,rfm_score,rfm_segment
0,12346,326,1,77183.60,77183.600,74215,1,77183.600000,2011-01-18 10:01:00,2011-01-18 10:01:00,0,30.000000,74215.000000,77183.600000,0.000560,1,1,5,7,Medium Value
1,12347,2,7,4310.00,4166.600,2458,103,23.681319,2010-12-07 14:57:00,2011-12-07 15:52:00,365,0.573770,351.142857,615.714286,0.057639,5,5,5,15,Top Value
2,12348,75,4,1797.24,1770.240,2341,22,57.975484,2010-12-16 19:09:00,2011-09-25 13:13:00,282,0.424028,585.250000,449.310000,0.012311,2,4,4,10,High Value
3,12349,19,1,1757.55,1587.486,631,73,24.076027,2011-11-21 09:51:00,2011-11-21 09:51:00,0,30.000000,631.000000,1757.550000,0.040851,4,1,4,9,Medium Value
4,12350,310,1,334.40,332.032,197,17,19.670588,2011-02-02 16:01:00,2011-02-02 16:01:00,0,30.000000,197.000000,334.400000,0.009513,1,1,2,4,Low Value
